In [ ]:
from pathlib import Path
import sys
import csv
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from models.unet import UNet
from models.pretrained_segformer import SegFormer
from models.multi_attention_unet import MutliRepUnet
from src.lidar_dataset import LidarDataset
from src.metrics import segmentation_metrics, calculate_metrics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MANIFEST_PATH = PROJECT_ROOT / 'data/processed/manifest_split.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 2
NUM_WORKERS = 0
CLASS_NAMES = ['Building', 'Platform', 'Aguada']
MODEL_NAMES = ['U-Net', 'SegFormer', 'Multi-Rep U-Net']
CHECKPOINTS = {
    'U-Net': PROJECT_ROOT / 'checkpoints/ unet_combined.pt',
    'SegFormer': PROJECT_ROOT / 'checkpoints/ Segformer_b0.pt',
    'Multi-Rep U-Net': PROJECT_ROOT / 'checkpoints/multi_rep_unet.pt',
}
THRESHOLDS = {
    'U-Net': [0.4, 0.425, 0.525],
    'SegFormer': [0.425, 0.4, 0.5],
    'Multi-Rep U-Net': [0.35, 0.375, 0.3],
}
PR_THRESHOLDS = torch.linspace(0, 1, 101)
EXAMPLES_PER_CATEGORY = 1

In [ ]:
def save_and_show(fig, filename):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved:', OUTPUT_DIR / filename)


def build_model(name):
    if name == 'U-Net':
        return UNet(in_channels=3, out_channels=3)
    if name == 'SegFormer':
        return SegFormer(out_channels=3)
    if name == 'Multi-Rep U-Net':
        return MutliRepUnet(out_channels=3)
    raise ValueError(name)


def load_model(name):
    path = CHECKPOINTS[name]
    if not path.is_file():
        raise FileNotFoundError(f'{name} checkpoint missing: {path}')
    model = build_model(name).to(DEVICE)
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f'{name}: loaded {path.name}; epoch {checkpoint.get("epoch", "unknown")}')
    return model

In [ ]:
def choose_example_indices(dataset):
    #Choose first ground-truth-positive example for each class, plus background-only file.
    selected = {}
    for index in range(len(dataset)):
        _, mask = dataset[index]
        present = (mask > 0.5).reshape(3, -1).any(dim=1).tolist()
        for c, name in enumerate(CLASS_NAMES):
            if present[c] and name not in selected:
                selected[name] = index
        if not any(present) and 'Background' not in selected:
            selected['Background'] = index
        if len(selected) == 4:
            break
    print('Example tile indices:', selected)
    return selected


def normalize_for_display(image):
    #Normalize image for display
    x = image.detach().cpu().numpy().astype(np.float32)
    x = np.moveaxis(x, 0, -1)
    for c in range(3):
        lo, hi = np.percentile(x[..., c], [2, 98])
        x[..., c] = np.clip((x[..., c] - lo) / max(hi - lo, 1e-6), 0, 1)
    return x


def color_mask(mask):
    colors = np.array([[1.0, 0.3, 0.3], [0.2, 0.85, 0.3], [0.25, 0.55, 1.0]])
    mask = np.asarray(mask, dtype=bool)
    rgb = np.zeros((*mask.shape[1:], 3), dtype=np.float32)
    n = np.zeros(mask.shape[1:], dtype=np.float32)
    for c in range(3):
        rgb += mask[c, ..., None] * colors[c]
        n += mask[c]
    return np.divide(rgb, np.maximum(n[..., None], 1), out=np.zeros_like(rgb), where=n[..., None] > 0)


def error_overlay(image, predicted, target):
    #Red = false positive, blue = false negative, green = true positive.
    fp = predicted & ~target
    fn = ~predicted & target
    tp = predicted & target
    rgb = normalize_for_display(image) * 0.40
    rgb[tp] = [0.1, 0.95, 0.2]
    rgb[fp] = [1.0, 0.1, 0.1]
    rgb[fn] = [0.15, 0.45, 1.0]
    return rgb

In [ ]:
@torch.no_grad()
def evaluate_model(name, model, loader, chosen_indices):
    tp = torch.zeros(3, device=DEVICE)
    fp = torch.zeros(3, device=DEVICE)
    fn = torch.zeros(3, device=DEVICE)
    bins = 101
    hist_pos = np.zeros((3, bins), dtype=np.int64)
    hist_neg = np.zeros((3, bins), dtype=np.int64)
    examples = {}
    offset = 0

    for images, masks in tqdm(loader, desc=f'Evaluating {name}', unit='batch'):
        images = images.to(DEVICE, dtype=torch.float32)
        masks = masks.to(DEVICE, dtype=torch.float32)
        logits = model(images)
        counts = segmentation_metrics(logits, masks, threshold=THRESHOLDS[name])
        tp += counts['tp']
        fp += counts['fp']
        fn += counts['fn']
        probs = torch.sigmoid(logits)
        targets = masks > 0.5

        bin_ids = (probs * (bins - 1)).long().clamp(0, bins - 1)
        for c in range(3):
            pos_bins = bin_ids[:, c][targets[:, c]]
            neg_bins = bin_ids[:, c][~targets[:, c]]
            hist_pos[c] += torch.bincount(pos_bins, minlength=bins).cpu().numpy()
            hist_neg[c] += torch.bincount(neg_bins, minlength=bins).cpu().numpy()

        for local in range(images.shape[0]):
            idx = offset + local
            if idx in chosen_indices.values():
                examples[idx] = {
                    'image': images[local].cpu(),
                    'target': targets[local].cpu().numpy(),
                    'probs': probs[local].cpu().numpy(),
                }
        offset += images.shape[0]

    metrics = calculate_metrics(tp, fp, fn)
    return metrics, hist_pos, hist_neg, examples